# RAG Pipeline Test
## Testing the full pipeline: PDF → Chunks → Embeddings → ChromaDB → LLaMA Answer

In [ ]:
import sys
sys.path.append('../backend')

from dotenv import load_dotenv
load_dotenv('../.env')
print("Environment loaded")

In [ ]:
from app.utils.chunker import load_and_chunk_text

# Test with sample text instead of a real PDF first
sample_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines. 
LLaMA 3 is Meta's open source large language model released in 2024.
It has 8 billion parameters in its smallest version.
RAG stands for Retrieval Augmented Generation. It combines 
a retrieval system with a language model to produce accurate answers.
ChromaDB is an open source vector database used to store embeddings.
"""

chunks = load_and_chunk_text(sample_text, source_name="test_document")
print(f"Created {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(chunk.page_content[:100])

In [ ]:
from app.services.embedder import embed_and_store, get_document_count, delete_vectorstore

# Clear any existing data first
delete_vectorstore()

# Embed and store the chunks
vectorstore = embed_and_store(chunks, collection_name="test")
count = get_document_count()
print(f"ChromaDB now has {count} chunks stored")

In [ ]:
from app.services.retriever import retrieve_relevant_chunks, format_context_for_prompt

query = "What is RAG and how does it work?"
results = retrieve_relevant_chunks(query, k=3)

context, sources = format_context_for_prompt(results)

print("RETRIEVED CONTEXT:")
print(context)
print("\nSOURCES:")
for source in sources:
    print(source)

In [ ]:
from app.services.llm import generate_answer_hf_api

question = "What is LLaMA 3 and who made it?"
answer = generate_answer_hf_api(question, context)

print(f"QUESTION: {question}")
print(f"\nANSWER: {answer}")
print(f"\nSOURCES USED: {[s['file'] for s in sources]}")

## Pipeline Test Complete!
If all cells ran without errors, your RAG pipeline is working correctly.
Next step: Start the FastAPI server and test via the API.